In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

In [ ]:
using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using GeneralizedPerturbedEquilibrium: InnerLayer
using GeneralizedPerturbedEquilibrium.InnerLayer: solve_inner
using GeneralizedPerturbedEquilibrium: Tearing
using ..InnerLayer
using ..InnerLayer: InnerLayerModel, solve_inner, GGJModel, GGJParameters,
    SLAYERModel, SLAYERParameters

using Plots
using Printf
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
struct TorqueBalance{M<:InnerLayerModel,P}
    model::M
    params::P
    Q0::Float64
    P::Float64
    lu::Float64
    sval::Float64
end

function torque_balance_value(tb::TorqueBalance, Q::Number)
    Δ = solve_inner(tb.model, tb.params, ComplexF64(Q)).tearing
    alpha = 1e-2
    jxb = -imag(1.0 / (Δ + alpha))
    return 2.0 * tb.P * (tb.Q0 - Q) / jxb, Δ
end

function torque_balance_scan(tb; Qmin=-10.0, Qmax=10.0, n=200)
    Qs = range(Qmin, Qmax; length=n)
    torque_out = [torque_balance_value(tb, q) for q in Qs]
    bal = [x[1] for x in torque_out]
    Δs = [x[2] for x in torque_out]
    #bal = [torque_balance_value(tb, q) for q in Qs]
    positive = isfinite.(bal) .& (bal .> 0.0)

    if !any(positive)
        return Qs, bal, NaN, NaN, 0.0, NaN, NaN, Δs
    end

    i = argmax(bal)
    Qpeak_ind = i
    Qs_positive = Qs[positive]
    bal_positive = bal[positive]


    
    Qpeak = Qs_positive[i]
    maxbal = bal_positive[i]

    br_crit = sqrt(maxbal / tb.lu * (tb.sval^2 / 2.0))
    return Qs, bal, Qs_positive, bal_positive, Qpeak, br_crit, Qpeak_ind, Δs
end

In [ ]:
p = GeneralizedPerturbedEquilibrium.InnerLayer.slayer_parameters(
    n_e=1e19, t_e=1e3, t_i=1e3,
    omega=0.0, omega_e=40, omega_i=-20,
    qval=2.0, sval_r=0.5, bt=2.0, rs=1.0, R0=3.0, mu_i=2.0, zeff=1.0,
    chi_perp=1.0, chi_tor=1.0, m=2, n=1
)
p = GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERParameters(
    ising=p.ising,
    m=p.m, n=p.n,
    tau=p.tau, lu=p.lu, c_beta=p.c_beta, D_norm=p.D_norm,
    P_perp=p.P_perp, P_tor=p.P_tor,
    Q_e=2, Q_i=3, iota_e=p.iota_e,
    tauk=p.tauk, tau_r=p.tau_r, delta_n=p.delta_n,
    rs=p.rs, R0=p.R0, bt=p.bt, sval_r=p.sval_r,
    dr_val=p.dr_val, dgeo_val=p.dgeo_val,
    eta=p.eta, d_beta=p.d_beta,
    dc_tmp=p.dc_tmp, dc_type=p.dc_type
)

tb = TorqueBalance(
    GeneralizedPerturbedEquilibrium.InnerLayer.SLAYERModel(;),
    p,
    0.5,
    61.0, #1.0
    p.lu,
    p.sval_r
)

In [ ]:

Qs, bal, Qs_positive, bal_positive, Qpeak, brcrit, Qpeak_ind, Δs = torque_balance_scan(tb,Qmin=-2.5, Qmax=0, n=20000)

jxbs = [-imag(1.0 / (d + 1e-2)) for d in Δs]
T_VISC = [2.0 * tb.P * (tb.Q0 - q) for q in Qs] #(q, jxb) in zip(Qs, jxbs)]
#br_t = brcrit
#println(tb.lu, " ", tb.sval, " ", br_t, " ", p.bt, " ", 1e-2)
#@printf("br_crit = %.5e\n", brcrit)
#T_EM = tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs
T_EM = -imag(1.0 ./ (Δs .+ 1e-2)) *brcrit^2*tb.lu/(tb.sval^2/2)
i = Qpeak_ind
println(T_VISC[i]/(tb.lu * tb.sval^2/2 * (brcrit)^2 * jxbs[i]))
println(T_VISC[i] / (tb.lu * tb.sval^2/2 * (brcrit * p.bt)^2 * jxbs[i]))
println(T_VISC[i] / ( (brcrit / p.bt)^2 * jxbs[i]))

#T_EM ∝ S ξ̂ (br/Bφ)² Im[-Δ̂(Q)⁻¹]
#T_visc ∝ 2 P (Q0 − Q)

xmi = -2.5
xma = 0

"""
p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2, xlim=(xmi, xma))
#plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
xlabel!(p1, "Q")
ylabel!(p1, "Δ")
#title!(p1, "Inner-layer Δ(Q)")
"""

p1 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(-1000,1000))
plot!(p1, Qs, T_EM, label="T_EM", lw=2)
vline!([Qpeak], label="peak Q", linestyle=:dash)
xlabel!(p1, "Q")
ylabel!(p1, "Torque")

#p2 = plot(Qs, jxbs, label="jxb", lw=2)
p2 = plot( Qs, T_VISC, label="T_V", lw=2, xlim=(xmi, xma))#,ylim=(0,500))
plot!(p2, Qs, T_EM, label="T_EM", lw=2)
vline!([Qpeak], label="peak Q", linestyle=:dash)
xlabel!(p2, "Q")
ylabel!(p2, "Torque")
#title!(p2, "jxb(Q) = -Im[1/(Δ + δ_n_p)]")

p3 = plot(Qs, real.(bal), label="Re(balance)", lw=2, xlim=(xmi, xma))
plot!(p3, Qs, imag.(bal), label="Im(balance)", lw=2)
ylabel!(p3, "Torque Balance")
#title!(p3, "2P(Q0-Q)/jxb")
xlabel!(p3, "Q")
vline!([Qpeak], label="Max Balance", linestyle=:dash)

#plot(p1, p2, p3, layout=(3,1), size=(800, 1000))
plot(p3, layout=(1,1), size=(800, 1000))

In [ ]:
# compare Q_e to Q of max(bal)
println("-Q_e = ", -p.Q_e, " Q_i = ", p.Q_i, " Q_peak = ", Qpeak, " br_crit = ", brcrit)
# check if |Qpeak+Q_e| < 1e-2
println("|Qpeak + Q_e| = ", abs(Qpeak + p.Q_e))

In [ ]:
# Check torque is balanced

viscous_torque = 2*tb.P*(tb.Q0 - Qpeak)
electromagnetic_torque = -imag(1.0 / (Δs[Qpeak_ind] + 1e-2)) *brcrit^2*tb.lu/(tb.sval^2/2)

# delta_n_p = 1e-2
# jxb = -imag(1.0 / (Δ + delta_n_p))
# bal = 2.0 * tb.P * (tb.Q0 - Q) / jxb
# br_crit = sqrt(maxbal / tb.lu * (tb.sval^2 / 2.0))

@printf("br_crit = %.5e\n", brcrit)
@printf("viscous_torque = %.5e\n", viscous_torque)
@printf("electromagnetic_torque = %.5e\n", electromagnetic_torque)
@printf("ratio = %.10f\n", viscous_torque / electromagnetic_torque)